## Ultracompact Waveguide Crossing via FMSA & Gaussian Upsampling

這個Tutorial主要是在說明如何利用 **因子分解機結合模擬退火/量子退火 (Factorization Machine Quantum/Simulated Annealing)** 演算法，進行**超緊湊雙模波導交叉器 (Ultra-compact dual-mode Waveguide Crossing)** 反向設計的原理、程式結構與執行方法。

本筆記本 (Notebook) 已經將整個專案的程式碼分解成數個邏輯區塊，並加上了詳細的解說。這能幫助您了解從「幾何結構生成」、「物理光學模擬」、「代理模型訓練」到「平行化演算法架構」的所有細節。

### 1. System Package 與 MPI 平行化設定
首先載入所需的FDTD光學模擬 (`meep`)、神經網路訓練框架 (`torch`)、最佳化與D-Wave的退火套件 (`dimod`)，以及科學計算庫。\
同時，這裡設定了 `mpi4py` 的 `comm` 變數。\
在多核心運作時，Master 節點會負責機器學習與退火，Worker 節點會負責平行的 MEEP 模擬。

In [ ]:
### 系統與基礎工具
import os
import time
import json
import argparse
from datetime import datetime

### 數學與科學計算
import numpy as np
import scipy.ndimage 

### 畫圖用
import matplotlib.pyplot as plt

### 平行計算
from mpi4py import MPI

### 神經網路架構PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

### 量子/模擬退火
import dimod
from dwave.samplers import SimulatedAnnealingSampler
try:
    from dwave.system import DWaveSampler, EmbeddingComposite
    DWAVE_AVAILABLE = True
except ImportError:
    DWAVE_AVAILABLE = False
    if MPI.COMM_WORLD.Get_rank() == 0:
        print("Warning: dwave-ocean-sdk not installed. QA mode will fail.")

### 光學模擬
import meep as mp

### 因子分解機 ( Factorization Machine )(自定義模組)
from factorization_machine import FactorizationMachine

In [ ]:
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

WORK_TAG = 1
STOP_TAG = 2

## 2. 全域物理與幾何結構參數定義
在這裡我們定義了 FDTD 的解析度 (`RESOLUTION`)、波導幾何尺寸 (`MDM_LC`, `WG_WIDTH`)、材料折射率、光源頻率波長頻寬設定，以及**高斯上採樣 (Gaussian Upsampling)** 的核心參數 (`KERNEL_SIGMA`, `TANH_BETA`)。

In [ ]:
RESOLUTION = 40     # 解析度 (波導約設定 20 或 40)
DPML = 1.0          # 吸收邊界反射的光波
MDM_LC = 2.4        # 設計區域邊長大小(Design Region)
WG_LENGTH = 1.0     # 波導長度
WG_WIDTH = 1.0      # 波導寬度

SX = 2 * DPML + MDM_LC + 2 * WG_LENGTH
SY = 2 * DPML + MDM_LC + 2 * WG_LENGTH
CELL = mp.Vector3(SX, SY, 0) #  整個 2D 模擬視窗的總大小

### 矽（核心）與二氧化矽（包層)折射率
N_SIO2 = 1.44
N_SI = 3.48
SIO2_MEDIUM = mp.Medium(index=N_SIO2)
SI_MEDIUM = mp.Medium(index=N_SI)

wl_cen = 1.55       # 模擬中心波長
FCEN = 1 / wl_cen   # 波長倒數(頻率)
DF = 0.1 * FCEN     # 高斯脈衝光源的頻寬
NFREQ = 1           # 設定波長中分幾個點

### Upsampling & Projection設定
KERNEL_SIZE = 5     # 高斯模糊卷積核的尺寸
KERNEL_SIGMA = 1.0  # 高斯模糊的標準差。Sigma 越大，邊角會越圓滑。
TANH_BETA = 50      # 雙曲正切投影的銳利度。數值越高，黑白邊界越分明，灰階過渡區域越少
TANH_ETA = 0.5      # 投影邊界的閾值
SMOOTH_THRESHOLD = 0.5

### 模擬退火設定深度(數值越大，花費時間可能越多)
NUM_SWEEPS = 1000

## 3. 對稱性展開與Gaussian Upsampling
在反向設計中，直接優化離散的二元像素會導致光場產生階梯效應且無法製造（違反最小特徵尺寸, MFS）。

這段程式碼的作用是將「數學上的 0 與 1」轉換成「物理上平滑且可製造的結構」：
1. **`gaussian_kernel` & `tanh_projection`**: 高斯濾波器能模糊零碎的邊角，接著利用 Tanh 投影將其銳化成平滑的 0 ($SiO_{2}$) 或 1 (Si) 結構。
2. **`enforce_reflection_symmetry`**: 為了使波導交叉器對稱且降低優化空間，演算法只生成「八分之一」的三角形矩陣區域，然後向左右、上下鏡射展開為完整的設計區域。
3. **`create_projected_geometry`**: 將最後的平滑矩陣轉換為 MEEP 的 `MaterialGrid` 物理區塊。
4. **`generate_smooth_random_config`**: 產生雲朵般平滑的初始隨機結構，避免給定毫無物理意義的純白噪音。

In [ ]:
def gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA):
    ax = np.arange(-(size//2), size//2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel /= np.sum(kernel)
    return kernel

def tanh_projection(x, beta=TANH_BETA, eta=TANH_ETA):
    num = np.tanh(beta * eta) + np.tanh(beta * (x - eta))
    den = np.tanh(beta * eta) + np.tanh(beta * (1 - eta))
    return num / den

def enforce_reflection_symmetry(matrix_quadrant):
    """
    將八分之一的矩陣 (Q) 擴充為具備四重對稱 (上下、左右、對角線) 的完整矩陣
    """
    Q = np.array(matrix_quadrant)
    Q_upper = np.triu(Q)                        # 強制對角線對稱
    Q_symmetric = Q_upper + np.triu(Q, 1).T     # 將上三角矩陣其翻轉疊加到下三角
    top_half = np.hstack((Q_symmetric, np.fliplr(Q_symmetric))) # 上下與左右鏡射展開
    bottom_half = np.hstack((np.flipud(Q_symmetric), np.flipud(np.fliplr(Q_symmetric))))
    
    return np.vstack((top_half, bottom_half))

def get_projected_density_matrix(binary_vector, grid_rows, grid_cols):
    vec_len = len(binary_vector)
    if vec_len == grid_rows * grid_cols:
        grid_matrix = np.array(binary_vector).astype(float).reshape((grid_rows, grid_cols))
    elif vec_len == (grid_rows // 2) * (grid_cols // 2):
        q_rows = grid_rows // 2
        q_cols = grid_cols // 2
        Q = np.array(binary_vector).astype(float).reshape((q_rows, q_cols))
        grid_matrix = enforce_reflection_symmetry(Q)
    else:
        raise ValueError(f"Invalid binary_vector size: {vec_len}")

    kernel = gaussian_kernel(size=KERNEL_SIZE, sigma=KERNEL_SIGMA)
    try:
        from scipy.signal import convolve2d
        density = convolve2d(grid_matrix, kernel, mode='same', boundary='symm')
    except Exception:
        k = kernel.shape[0]
        pad = k // 2
        img_p = np.pad(grid_matrix, ((pad, pad), (pad, pad)), mode='reflect')
        density = np.zeros_like(grid_matrix)
        for i in range(grid_rows):
            for j in range(grid_cols):
                density[i, j] = np.sum(img_p[i:i+k, j:j+k] * kernel)
    
    projected_density = tanh_projection(density, beta=TANH_BETA, eta=TANH_ETA)
    projected_density = np.clip(projected_density, 0.0, 1.0)
    return projected_density

def create_projected_geometry(binary_vector, grid_rows, grid_cols):
    density = get_projected_density_matrix(binary_vector, grid_rows, grid_cols)
    weights = density.flatten()
    material_grid = mp.MaterialGrid(mp.Vector3(grid_cols, grid_rows), SIO2_MEDIUM, SI_MEDIUM, weights=weights)
    material_grid.smoothing_radius = 1.0 
    design_block = mp.Block(size=mp.Vector3(MDM_LC, MDM_LC, mp.inf), center=mp.Vector3(), material=material_grid)
    return [design_block]

def generate_smooth_random_config(rows, cols):
    small_r, small_c = max(1, (rows + 1) // 2), max(1, (cols + 1) // 2) 
    noise = np.random.rand(small_r, small_c)
    smooth_noise = scipy.ndimage.zoom(noise, zoom=2.0, order=1) 
    smooth_noise = smooth_noise[:rows, :cols]
    binary = (smooth_noise > SMOOTH_THRESHOLD).astype(int).flatten()
    return binary

## 4. Fom Evaluate & Optical Simulation
這部分是負責計算波導的穿透率與FoM。\
首先會將前一個函數生成的「二元幾何圖形」，丟入 **MEEP** 中，實際跑一次電磁波模擬，並回傳這個波導交叉器的穿透率。
這邊我們基於Fom的需求，我們設定優化兩個模態(TE0、TE1)且三個目標波長[1.52, 1.56, 1.60]$\mu m$\

1. **架設實體結構**: 設定上、下、左、右四條固定的矽 (Si) 波導，並與中央的 `mdm_structure` 結合。
2. **架設雷射光源**: 透過 `EigenModeSource` 在左側輸入高斯寬頻脈衝，並根據 `mode_name` 指定為 TE0 偶對稱或 TE1 奇對稱。
3. **裝設功率感測器**: 在輸入端和直行輸出端 (東側) 設立 `ModeRegion` 監測器以量測功率。
4. **執行與提早停止**: 利用 `stop_when_fields_decayed`，當光波完全離開模擬區域（電場強度低於 $1e-4$）時就自動停止，節省時間。
5. **計算穿透率 (Transmission)**: 解析 S 參數，計算傳輸過去的能量百分比並回傳。

In [ ]:
def evaluate_mdm_mode(binary_vector, grid_rows, grid_cols, mode_name):
    mp.Simulation(cell_size=CELL, resolution=1, boundary_layers=[]).reset_meep()
    mdm_structure = create_projected_geometry(binary_vector, grid_rows, grid_cols)
    
    input_wg_center_x = -MDM_LC / 2 - (WG_LENGTH + DPML) / 2    # 左
    through_wg_center_x = MDM_LC / 2 + (WG_LENGTH + DPML) / 2   # 祐
    cross_top_center_y = MDM_LC / 2 + (WG_LENGTH + DPML) / 2    # 上
    cross_bot_center_y = -MDM_LC / 2 - (WG_LENGTH + DPML) / 2   # 下
    
    fixed_geometry = [
        # 左側波導 (Input)
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), 
                 center=mp.Vector3(input_wg_center_x, 0), material=SI_MEDIUM),
        # 右側波導 (Through)
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), 
                 center=mp.Vector3(through_wg_center_x, 0), material=SI_MEDIUM),
        # 上波導 (Cross Top)
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), 
                 center=mp.Vector3(0, cross_top_center_y), material=SI_MEDIUM),
        # 下波導 (Cross Bottom)
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), 
                 center=mp.Vector3(0, cross_bot_center_y), material=SI_MEDIUM),
    ]
    
    full_geometry = fixed_geometry + mdm_structure # 將Design Region跟四個波導結合
    
    src_center = mp.Vector3(-SX / 2 + DPML + 0.2, 0)    # 設定光源中心
    src_size = mp.Vector3(0, WG_WIDTH)                  # 設定光源尺寸大小
    mon_x_through = MDM_LC / 2 + WG_LENGTH / 2          # 規一化監視器的x座標
    mon_y_cross_top = MDM_LC / 2 + WG_LENGTH / 2        # 上波導串擾監視器的y座標
    mon_y_cross_bot = -MDM_LC / 2 - WG_LENGTH / 2       # 下波導串擾監視器的y座標
    monitor_size_y = mp.Vector3(0, WG_WIDTH * 5)        # 輸入輸出波導監視器尺寸大小
    monitor_size_x = mp.Vector3(WG_WIDTH * 5, 0)        # 串擾波導監視器大小
    
    ### 設定模態數與對稱性
    mode_props = {
        'TE0': {'band_num': 1, 'parity': mp.EVEN_Y, 'global_band': 1},
        'TE1': {'band_num': 2, 'parity': mp.ODD_Y, 'global_band': 2}
    }
    props = mode_props[mode_name]
    
    ### 設定光源
    sources = [mp.EigenModeSource(src=mp.GaussianSource(FCEN, fwidth=DF), 
                                  center=src_center, size=src_size, direction=mp.X, 
                                  eig_band=props['band_num'], eig_parity=props['parity'])]

    sim = mp.Simulation(cell_size=CELL, boundary_layers=[mp.PML(DPML)], 
                        geometry=full_geometry, sources=sources, 
                        resolution=RESOLUTION, default_material=SIO2_MEDIUM)

    ### 設定目標波長
    # 這邊與文獻相同，設定三個波長點
    target_wls = [1.52, 1.56, 1.60]
    target_freqs = [1/wl for wl in target_wls]

    ### 設定監視器
    """
    1. norm_fluxes為輸入監視器
    2. flux_throughs為輸出監視器
    """
    in_flux_region = mp.ModeRegion(center=mp.Vector3(src_center.x + 0.5, 0), size=monitor_size_y)
    norm_fluxes = [sim.add_mode_monitor(f, 0, 1, in_flux_region) for f in target_freqs]
    
    through_flux_region = mp.ModeRegion(center=mp.Vector3(mon_x_through, 0), size=monitor_size_y)
    flux_throughs = [sim.add_mode_monitor(f, 0, 1, through_flux_region) for f in target_freqs]
    
    sim.run(until_after_sources=mp.stop_when_fields_decayed(50, mp.Ez, mp.Vector3(mon_x_through, WG_WIDTH/4), 1e-4))
    
    T_through_list = []

    ### 迴圈計算Fom
    """
    每個模態分別有三個目標波長，因此總共會計算6個波長的穿透率，分別為:
    TE0 1.52 Transmission
    TE0 1.56 Transmission
    TE0 1.60 Transmission
    TE1 1.52 Transmission
    TE1 1.56 Transmission
    TE1 1.60 Transmission
    """
    for i in range(len(target_freqs)):
        input_flux = sim.get_eigenmode_coefficients(norm_fluxes[i], [props['band_num']], eig_parity=props['parity']).alpha[0, 0, 0]
        input_power = np.abs(input_flux)**2 + 1e-12
        
        through_coeff = sim.get_eigenmode_coefficients(flux_throughs[i], [props['band_num']], eig_parity=props['parity']).alpha[0, 0, 0]
        T_through = np.abs(through_coeff)**2 / input_power
        T_through_list.append(T_through)

    sim.reset_meep()
    return {'through': T_through_list}

## 5. Final Analysis & Plot
包含了大部分的繪圖功能與光學分析：
* **`plot_transmission_spectrum`**: 繪製寬頻譜的穿透率曲線 (dB vs Wavelength)。
* **`perform_detailed_final_analysis`**: 當演算法找到「最佳結構」後，會呼叫此函式進行一次非常詳細的模擬。這會啟動 DFT (Discrete Fourier Transform) 擷取完整的電場 $E_z$ 分佈，並畫出漂亮的光場強弱分佈圖。
* **`plot_fom_history` & `plot_optimization_trajectory`**: 繪製最佳化過程中，目標函數 (FOM) 隨著迭代下降的收斂曲線圖。
* **`plot_time_statistics_pie_chart`**: 繪製耗時圓餅圖（FM訓練 vs FDTD 模擬 vs 模擬退火）。

In [ ]:
### 繪製頻譜(Transmission vs. wavelength)
def plot_transmission_spectrum(wls, trans_dbs, mode_name, output_folder):
    from scipy.interpolate import make_interp_spline
    plt.figure(figsize=(6, 5))
    wls = np.array(wls)
    trans_dbs = np.array(trans_dbs)
    sorted_idx = np.argsort(wls)
    wls = wls[sorted_idx]
    trans_dbs = trans_dbs[sorted_idx]
    
    if len(wls) >= 3:
        x_smooth = np.linspace(wls.min(), wls.max(), 100)
        spline = make_interp_spline(wls, trans_dbs, k=2)
        y_smooth = spline(x_smooth)
        plt.plot(x_smooth, y_smooth, 'b-', linewidth=2, label='FMQA')
    else:
        plt.plot(wls, trans_dbs, 'b-', linewidth=2, label='FMQA')
        
    plt.title(f"Transmission Spectrum ({mode_name})", fontsize=16)
    plt.xlabel(r"wavelength ($\mu$m)", fontsize=14)
    plt.ylabel("Transmission (dB)", fontsize=14)
    plt.ylim([-2, 0])
    plt.grid(True, alpha=0.3)
    plt.legend(loc='lower right', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f"Transmission_Spectrum_{mode_name}.png"), dpi=300)
    plt.close()

### 最佳結構的光學分析，並記錄
def perform_detailed_final_analysis(best_config, grid_rows, grid_cols, output_folder):
    print(f"\n>>> Starting Final Detailed MEEP Analysis <<<")

    mdm_structure = create_projected_geometry(best_config, grid_rows, grid_cols)
    input_wg_center_x = -MDM_LC / 2 - (WG_LENGTH + DPML) / 2    # 左
    through_wg_center_x = MDM_LC / 2 + (WG_LENGTH + DPML) / 2   # 右
    cross_top_center_y = MDM_LC / 2 + (WG_LENGTH + DPML) / 2    # 上
    cross_bot_center_y = -MDM_LC / 2 - (WG_LENGTH + DPML) / 2   # 下
    
    fixed_geometry = [
        # 左側波導 (Input)
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), 
                 center=mp.Vector3(input_wg_center_x, 0), material=SI_MEDIUM),
        # 右側波導 (Through)
        mp.Block(size=mp.Vector3(WG_LENGTH + DPML + 0.1, WG_WIDTH, mp.inf), 
                 center=mp.Vector3(through_wg_center_x, 0), material=SI_MEDIUM),
        # 上側波導 (Cross Top)
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), 
                 center=mp.Vector3(0, cross_top_center_y), material=SI_MEDIUM),
        # 下側波導 (Cross Bottom)
        mp.Block(size=mp.Vector3(WG_WIDTH, WG_LENGTH + DPML + 0.1, mp.inf), 
                 center=mp.Vector3(0, cross_bot_center_y), material=SI_MEDIUM),
    ]
    
    full_geometry = fixed_geometry + mdm_structure
    
    src_center = mp.Vector3(-SX / 2 + DPML + 0.2, 0)
    src_size = mp.Vector3(0, WG_WIDTH) 
    mon_x_through = MDM_LC / 2 + WG_LENGTH / 2
    mon_y_cross_top = MDM_LC / 2 + WG_LENGTH / 2
    mon_y_cross_bot = -MDM_LC / 2 - WG_LENGTH / 2
    monitor_size_y = mp.Vector3(0, WG_WIDTH * 5)
    monitor_size_x = mp.Vector3(WG_WIDTH * 5, 0) 
    
    mode_definitions = ['TE0', 'TE1']
    detailed_results = {}
    
    ### 每個模態分別跑一次
    for mode_name in mode_definitions:
        mode_props = {
            'TE0': {'band_num': 1, 'parity': mp.EVEN_Y, 'global_band': 1},
            'TE1': {'band_num': 2, 'parity': mp.ODD_Y, 'global_band': 2}
        }
        props = mode_props[mode_name]
        
        ### 光源設定
        sources = [mp.EigenModeSource(src=mp.GaussianSource(FCEN, fwidth=DF), 
                                      center=src_center, size=src_size, direction=mp.X, 
                                      eig_band=props['band_num'], eig_parity=props['parity'])]
        sim = mp.Simulation(cell_size=CELL, boundary_layers=[mp.PML(DPML)], 
                        geometry=full_geometry, sources=sources, 
                        resolution=RESOLUTION, default_material=SIO2_MEDIUM)
        
        ### 設定目標波長
        target_wls = [1.52, 1.56, 1.60]
        target_freqs = [1/wl for wl in target_wls]

        in_flux_region = mp.ModeRegion(center=mp.Vector3(src_center.x + 0.5, 0), size=monitor_size_y)
        norm_fluxes = [sim.add_mode_monitor(f, 0, 1, in_flux_region) for f in target_freqs]
        through_flux_region = mp.ModeRegion(center=mp.Vector3(mon_x_through, 0), size=monitor_size_y)
        flux_throughs = [sim.add_mode_monitor(f, 0, 1, through_flux_region) for f in target_freqs]
        
        ### DFT光場設定
        dft_monitor = sim.add_dft_fields([mp.Ez], FCEN, FCEN, 1, center=mp.Vector3(), size=mp.Vector3(SX, SY))
        
        ### 執行模擬
        sim.run(until_after_sources=mp.stop_when_fields_decayed(50, mp.Ez, mp.Vector3(mon_x_through, WG_WIDTH/4), 1e-4))
        
        ### 取 Ez分量 的DFT光場分布
        ez_dft_data = sim.get_dft_array(dft_monitor, mp.Ez, 0)
        eps_data = sim.get_epsilon()
        intensity = np.abs(ez_dft_data)**2
        
        T_through_list = []
        ### 計算最後一次最佳 Fom
        for i in range(len(target_freqs)):
            res_input = sim.get_eigenmode_coefficients(norm_fluxes[i], [props['band_num']], eig_parity=props['parity']).alpha[0, 0, 0]
            input_power = np.abs(res_input)**2 + 1e-12
            
            through_coeff = sim.get_eigenmode_coefficients(flux_throughs[i], [props['band_num']], eig_parity=props['parity']).alpha[0, 0, 0]
            T_through = np.abs(through_coeff)**2 / input_power
            T_through_list.append(T_through)

        detailed_results[mode_name] = {}
        trans_dbs_plot = []
        ### 紀錄模擬結果至Json
        for idx, wl in enumerate(target_wls):
            trans = T_through_list[idx]
            trans_db = 10 * np.log10(trans + 1e-9)
            trans_dbs_plot.append(trans_db)
            print(f"  [Result] {mode_name} L->R (Through) @ {wl}$\mu$m: {trans:.4f} ({trans_db:.2f} dB)")
            detailed_results[mode_name][f"{wl}$\mu$m"] = [float(trans), float(trans_db)]
    
        plot_transmission_spectrum(target_wls, trans_dbs_plot, mode_name, output_folder)

        ### 繪製DFT光場分布與結構輪廓
        x = np.linspace(-SX/2, SX/2, intensity.shape[0])
        y = np.linspace(-SY/2, SY/2, intensity.shape[1])
        fig, ax = plt.subplots(figsize=(8, 8))
        im = ax.imshow(intensity.T, extent=[x.min(), x.max(), y.min(), y.max()], 
                   cmap='inferno', origin='lower')
        ax.contour(eps_data.T, extent=[x.min(), x.max(), y.min(), y.max()], 
                   levels=[(N_SI**2+N_SIO2**2)/2], colors='white', alpha=0.5, linewidths=1, origin='lower')
        from mpl_toolkits.axes_grid1 import make_axes_locatable
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(im, cax=cax, label=r'Intensity $|E_z|^2$')
        ax.set_title(f"Optimized Final Structure ({mode_name})", fontsize=16)
        ax.set_xlabel(r"x ($\mu$m)", fontsize=14)
        ax.set_ylabel(r"y ($\mu$m)", fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"Optimized_Final_Structure_{mode_name}.png"), dpi=300)
        plt.close(fig)

    ### 繪製二元的平滑化結構分布
    x_mask = (x >= -MDM_LC/2) & (x <= MDM_LC/2)
    y_mask = (y >= -MDM_LC/2) & (y <= MDM_LC/2)
    eps_design = eps_data[np.ix_(x_mask, y_mask)]
    
    threshold_eps = (N_SI**2 + N_SIO2**2) / 2
    binary_design = (eps_design > threshold_eps).astype(int)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(binary_design.T, extent=[-MDM_LC/2, MDM_LC/2, -MDM_LC/2, MDM_LC/2], 
              cmap='gray_r', origin='lower')
    ax.set_title("Smoothed Binary Structure", fontsize=16)
    ax.set_xlabel(r"x ($\mu$m)", fontsize=14)
    ax.set_ylabel(r"y ($\mu$m)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "Smoothed_Binary_Structure.png"), dpi=300)
    plt.close(fig)

    return detailed_results

### 繪製最佳Fom軌跡圖
def plot_fom_history(fom_history, output_dir):
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(fom_history)), fom_history, marker='o', linestyle='-', color='b')
    plt.title("Best FOM vs Iteration", fontsize=16)
    plt.xlabel("Iteration", fontsize=14)
    plt.ylabel("Best FOM", fontsize=14)
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, "fom_evolution.png"), dpi=300)
    plt.close()

### 繪製所有Dataset vs Fom 的軌跡圖
def plot_optimization_trajectory(foms, split_index, adding_num, output_folder):
    plt.figure(figsize=(12, 6))
    
    iterations = np.zeros(len(foms))
    iterations[:split_index] = 0
    if len(foms) > split_index:
        opt_indices = np.arange(len(foms) - split_index)
        iterations[split_index:] = (opt_indices // adding_num) + 1
    
    plt.scatter(iterations[:split_index], foms[:split_index], 
                s=5, c='red', alpha=0.6, label='Initial Random Samples')
                
    if len(foms) > split_index:
        plt.scatter(iterations[split_index:], foms[split_index:], 
                    s=5, c='blue', alpha=0.6, label='FMQA Optimized Samples')
    
    if len(foms) > split_index:
        plt.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Opt Start')
    
    plt.xlabel('Iteration', fontsize=14)
    plt.ylabel('Optimization FOM', fontsize=14)
    plt.title('Optimization Trajectory', fontsize=16)
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    ax = plt.gca()
    max_iter = int(np.max(iterations))
    if max_iter > 0:
        step = max(1, max_iter // 10)
        ax.set_xticks(np.arange(0, max_iter + 1, step))
        
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "fom_trajectory.png"), dpi=150)
    plt.close()

### 繪製時間花費圓餅圖
def plot_time_statistics_pie_chart(total_fm, total_sa, total_fdtd, other, output_folder):
    labels = ['FM Training', 'Annealing Sampling', 'FDTD Simulation', 'Other']
    sizes = [total_fm, total_sa, total_fdtd, max(0, other)]
    all_colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99']
    
    total = sum(sizes)
    filtered_sizes = []
    filtered_labels = []
    filtered_colors = []
    for s, l, c in zip(sizes, labels, all_colors):
        if s > 0.1:
            filtered_sizes.append(s)
            pct = (s / total) * 100 if total > 0 else 0
            filtered_labels.append(f"{l} ({pct:.1f}%)")
            filtered_colors.append(c)
            
    fig, ax = plt.subplots(figsize=(10, 8))
    explode = [0.01] * len(filtered_sizes)
    
    def my_autopct(pct):
        return f'{pct:.1f}%' if pct > 3 else ''
        
    wedges, texts, autotexts = ax.pie(filtered_sizes, 
                                      autopct=my_autopct, 
                                      startangle=140, 
                                      colors=filtered_colors,
                                      explode=explode,
                                      textprops={'fontsize': 16})
    plt.setp(autotexts, size=16, weight="bold")
    
    lgd = ax.legend(wedges, filtered_labels, title="Runtime Components", 
                    loc="center left", bbox_to_anchor=(0.9, 0.5), fontsize=14, title_fontsize=16)

    ax.set_title("Total Runtime Distribution", fontsize=18)
    plt.savefig(os.path.join(output_folder, "time_distribution.png"), 
                dpi=300, bbox_extra_artists=(lgd,), bbox_inches='tight')
    plt.close()

## 6. 訓練因子分解機代理模型 (Factorization Machine)
這裡使用 PyTorch 實作了代理模型的訓練迴圈。\
每次有了新的 FDTD 模擬數據 (Config $\to$ FOM)，我們就會呼叫 `train_fm_model` 訓練一個 Factorization Machine (FM)。\
FM 能夠學習結構向量中各像素的「一階權重」與「二階交互作用 (像素間的關聯)」，預測此模型的權重應該為多少才能建立出好的QUBO Matrix分布。\
可以使後續的退火演算法可以有效採樣到更好的結構分布。

In [ ]:
def train_fm_model(model, X_train, Y_train, num_epoch, learning_rate, batch_size=32):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.MSELoss()
    device = next(model.parameters()).device
    
    X_tensor = torch.from_numpy(X_train).float().to(device)
    y_mean = Y_train.mean()
    y_std = Y_train.std() + 1e-8
    Y_scaled = (Y_train - y_mean) / y_std
    Y_tensor = torch.from_numpy(Y_scaled).float().view(-1, 1).to(device)
    
    dataset = TensorDataset(X_tensor, Y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    loss_history = []
    start_time = time.time()
    model.train() 
    
    print(f"  [FM Training] Dataset Size: {len(X_train)} samples")
    
    for epoch in range(num_epoch):
        epoch_loss = 0.0
        for batch_x, batch_y in dataloader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch_x.size(0)
            
        avg_loss = epoch_loss / len(dataset)
        loss_history.append(avg_loss)
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 100 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"    - Epoch [{epoch+1:3d}/{num_epoch}], Loss: {avg_loss:.6f}, LR: {current_lr:.2e}")
            
    return model, loss_history, time.time() - start_time, (y_mean, y_std)

## 7. MPI 平行化架構設定與退火迴圈
這是整個反向設計的「大腦」與「指揮中心」：

##### `worker_node` (工作節點)
會進入一個無窮迴圈，等待 Master 分派 FDTD 模擬任務。收到任務後執行 `evaluate_mdm_mode`，算完再把結果回傳。

In [ ]:
def worker_node():
    grid_dims = comm.bcast(None, root=0)
    mp.verbosity(0)
    while True:
        status = MPI.Status()
        data = comm.recv(source=0, tag=MPI.ANY_TAG, status=status)
        if status.Get_tag() == STOP_TAG: break
        task_idx, config, mode_name = data
        res = evaluate_mdm_mode(config, grid_dims['rows'], grid_dims['cols'], mode_name)
        comm.send((task_idx, mode_name, res), dest=0, tag=WORK_TAG)

##### `parallel_evaluate` (任務分派)
負責將一包要評估的結構分送給各個 Worker，並帶有快取 (Cache) 機制以避免重複計算相同的結構。\
並使每個空閒的CPU核心自動去找尚未進行模擬的結構進行模擬。\
使優化時間最小化。

In [ ]:
def parallel_evaluate(tasks, fom_cache, grid_rows, grid_cols):
    num_tasks = len(tasks)
    results = [None] * num_tasks
    partial_results = [{} for _ in range(num_tasks)]
    tasks_to_run = []
    
    for i, config in enumerate(tasks):
        config_tuple = tuple(config)
        if config_tuple in fom_cache: 
            results[i] = fom_cache[config_tuple]
        else: 
            tasks_to_run.append((i, config, 'TE0'))
            tasks_to_run.append((i, config, 'TE1'))

    if not tasks_to_run: return results

    num_to_run = len(tasks_to_run)
    sent_jobs = 0
    jobs_done = 0
    for worker_rank in range(1, min(size, num_to_run + 1)):
        comm.send(tasks_to_run[sent_jobs], dest=worker_rank, tag=WORK_TAG)
        sent_jobs += 1

    while jobs_done < num_to_run:
        status = MPI.Status()
        task_idx, mode_name, res_dict = comm.recv(source=MPI.ANY_SOURCE, tag=WORK_TAG, status=status)
        partial_results[task_idx][mode_name] = res_dict
        
        if 'TE0' in partial_results[task_idx] and 'TE1' in partial_results[task_idx]:
            trans_res = partial_results[task_idx]
            
            TE0_T = np.array(trans_res['TE0']['through'])
            TE1_T = np.array(trans_res['TE1']['through'])
            
            # 平均穿透率 (越接近 1 越好)
            avg_T = np.mean(np.concatenate([TE0_T, TE1_T]))
            
            # 因為演算法尋找最小值，所以設定為 1 - avg_T
            fom = 1.0 - avg_T
            
            results[task_idx] = fom
            fom_cache[tuple(tasks[task_idx])] = fom
            
        jobs_done += 1
        if sent_jobs < num_to_run:
            comm.send(tasks_to_run[sent_jobs], dest=status.Get_source(), tag=WORK_TAG)
            sent_jobs += 1
            
    return results

##### `master_node` (指揮節點核心迴圈)
1. 產生初始資料集 (`INIT_DATASET_SIZE`)。
2. 進入迴圈：
   - 訓練 FM 代理模型。
   - 提取 FM 的一階與二階權重 ($h$, $Q$) 構建 Binary Quadratic Model (BQM)。
   - 使用 **Simulated Annealing (或 D-Wave Quantum Annealing)** 在 $2^{144}$ 的龐大空間中找尋預期 FOM 最低（最佳）的新結構。
   - 分派結構給 Worker 進行 FDTD 驗證，並將結果加回資料集。
3. 輸出最後的 `best_config` 並進行 `perform_detailed_final_analysis`。

In [ ]:
def master_node():
    ### 優化參數設定
    PARAMS = {
        'GRID_ROWS': 12,            # 網格數
        'GRID_COLS': 12,            # 網格數(12 x 12 組成的三角形，總共 78 個 bits)
        'INIT_DATASET_SIZE': 1000,  # 初始DataSet
        'ITERATIONS': 100,          # 迭代數
        'ADDING_NUM': 30,           # 每次迭代新增模擬的Data數 / 退火採樣後取30個最佳結構
        'NUM_EPOCHS': 1000,         # Factorization Machine的訓練迭代次數
        'LEARNING_RATE': 1.0e-3,    # Factorization Machine訓練的步長
        'NUM_READS': 1500,          # SA/QA 的 讀取次數/採樣次數
        'K_FACTOR': 8,              # 潛在向量維度，決定代理模型學習能力與抗過擬合能力的核心參數
        'SAMPLER_TYPE': 'SA'        # 選擇QA/SA
    }
    parser = argparse.ArgumentParser(description="FMQA Crossing Inverse Design")
    parser.add_argument('--name', type=str, default=f'Crossing_waveguide_{PARAMS["INIT_DATASET_SIZE"]}_{PARAMS["ITERATIONS"]}x{PARAMS["ADDING_NUM"]}', help='Job Name')
    args = parser.parse_args()
    job_name = args.name
    total_meep_time = 0.0
    start_total = time.time()

    # 判斷 PyTorch 是否能使用 GPU 加速 FM 訓練
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"--- Master Node Started on {device} (Job: {job_name}) ---")

    # 計算全域物理維度與總變數數量
    Q_ROWS = PARAMS['GRID_ROWS']
    Q_COLS = PARAMS['GRID_COLS']
    FULL_ROWS = Q_ROWS * 2
    FULL_COLS = Q_COLS * 2
    NUM_VARS = Q_ROWS * Q_COLS
    
    # [目錄與存檔初始化] 建立帶有時間戳記的專屬資料夾，方便日後追蹤實驗軌跡與提取數據
    BASE_PATH = os.getcwd() 
    RESULTS_BASE_DIR = os.path.join(BASE_PATH, "Results_Crossing")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir_name = f"{job_name}_{timestamp}"
    output_folder = os.path.join(RESULTS_BASE_DIR, run_dir_name)
    os.makedirs(output_folder, exist_ok=True)
    
    # [MPI 同步] 將完整維度廣播給所有 Worker 節點，確保大家建立的 MEEP Cell 大小一致
    comm.bcast({'rows': FULL_ROWS, 'cols': FULL_COLS}, root=0)
    
    print(f"--- Generating Initial Dataset ({PARAMS['INIT_DATASET_SIZE']}) ---")

    # 生成具備平滑度限制的隨機二元結構，避免產生不符合製程規範 (如微影極限) 的破碎圖案
    configs = np.array([
        generate_smooth_random_config(Q_ROWS, Q_COLS) 
        for _ in range(PARAMS['INIT_DATASET_SIZE'])
    ])
    
    fom_cache = {}      # 建立快取機制，避免相同結構重複進入 FDTD 模擬浪費算力
    t_init_meep_start = time.time()

    # [平行 FDTD 評估] 派發任務給 MPI Worker 呼叫 MEEP 進行模擬，並等待全數回傳
    foms = np.array(parallel_evaluate(configs, fom_cache, FULL_ROWS, FULL_COLS))
    for i, c in enumerate(configs): fom_cache[tuple(c)] = foms[i]
    total_meep_time += time.time() - t_init_meep_start
    
    best_fom = np.min(foms)
    best_config = configs[np.argmin(foms)]
    print(f"Initial Best FOM: {best_fom:.4f}")

    # 初始化各項效能指標的紀錄器
    history = {'chain_break_trends': {'avg': [], 'max': []}, 'timing_metrics': {'fm_train_time': [], 'new_data_sim_time': [], 'sa_time': []}}
    best_fom_history = [best_fom] 

    # 初始化 Factorization Machine 模型
    model = FactorizationMachine(input_size=NUM_VARS, factorization_size=PARAMS['K_FACTOR']).to(device)

    for i in range(PARAMS['ITERATIONS']):
        print(f"\n=== Iteration {i+1}/{PARAMS['ITERATIONS']} ===")
        
        # 1. [代理模型訓練] 讓 FM 學習目前所有收集到的 (結構 -> 穿透率) 映射關係
        model, _, t_train, _ = train_fm_model(model, configs, foms, PARAMS['NUM_EPOCHS'], PARAMS['LEARNING_RATE'])
        history['timing_metrics']['fm_train_time'].append(t_train)
        
        # 2. [BQM] 提取 FM 的一階權重(h)與二階交互作用權重(Q)，轉換為退火所需的目標函數
        bias, h, Q = model.get_bhQ() 
        Q_dict = {(r, c): Q[r, c] for r in range(Q.shape[0]) for c in range(r+1, Q.shape[1]) if Q[r, c] != 0}
        bqm = dimod.BinaryQuadraticModel(h, Q_dict, bias, dimod.BINARY)
        
        # 3. [退火採樣] 根據參數決定使用 D-Wave 量子硬體或本地端 CPU 模擬退火
        t_sa_start = time.time()
        sampleset = None
        if PARAMS['SAMPLER_TYPE'] == "QA" and DWAVE_AVAILABLE:
            try:
                # 呼叫 D-Wave Advantage 系統
                sampler = EmbeddingComposite(DWaveSampler(solver='Advantage_system4.1'))
                sampleset = sampler.sample(bqm, num_reads=PARAMS['NUM_READS'], label=f"{job_name}_{i}")
                if 'chain_break_fraction' in sampleset.record.dtype.names:
                     history['chain_break_trends']['avg'].append(float(np.mean(sampleset.record['chain_break_fraction'])))
            except:
                # 若 QA 連線失敗或配額耗盡，自動降級為 SA，確保迴圈不中斷
                print("QA failed, using SA")
                sampler = SimulatedAnnealingSampler()
                sampleset = sampler.sample(bqm, num_reads=PARAMS['NUM_READS'], num_sweeps=NUM_SWEEPS)
        else:
            sampler = SimulatedAnnealingSampler()
            sampleset = sampler.sample(bqm, num_reads=PARAMS['NUM_READS'], num_sweeps=NUM_SWEEPS)

        sampleset = sampleset.aggregate()

        # 將退火得到的解依照能量 (預測的 FOM) 由小到大排序 
        sample_configs = sampleset.record['sample'][np.argsort(sampleset.record['energy'])]
        history['timing_metrics']['sa_time'].append(time.time() - t_sa_start)
        
        # 4. [探索機制防呆] 過濾掉已經存在於歷史資料集中的解，只保留「全新的設計」
        unique_new = [s for s in sample_configs if tuple(s) not in {tuple(x) for x in configs}]
        new_configs_list = unique_new[:PARAMS['ADDING_NUM']]
        
        # 如果退火找出的不重複新解數量不足 ADDING_NUM，自動用平滑隨機結構補齊，維持演算法的探索力 (Exploration)
        shortfall = PARAMS['ADDING_NUM'] - len(new_configs_list)
        if shortfall > 0:
            print(f"  [Notice] Sampler only yielded {len(new_configs_list)} unique configs. Auto-filling {shortfall} random configs.")
            existing_set = {tuple(x) for x in configs} | {tuple(x) for x in new_configs_list}
            attempts = 0
            while len(new_configs_list) < PARAMS['ADDING_NUM'] and attempts < shortfall * 10:
                rand_c = generate_smooth_random_config(Q_ROWS, Q_COLS)
                if tuple(rand_c) not in existing_set:
                    new_configs_list.append(rand_c)
                    existing_set.add(tuple(rand_c))
                attempts += 1
                
        new_configs = np.array(new_configs_list)
        
        # 5. [真實物理驗證] 將選出的高潛力候選結構送進 MEEP 模擬
        if new_configs.size > 0:
            num_new_samples = len(new_configs)
            print(f"  Evaluating {num_new_samples} new candidates...")
            
            t_sim_start = time.time()
            new_foms = parallel_evaluate(new_configs, fom_cache, FULL_ROWS, FULL_COLS)
            sim_duration = time.time() - t_sim_start
            total_meep_time += sim_duration

            # 將新結果併入歷史資料庫，作為下一次迭代的訓練養分
            configs = np.vstack([configs, new_configs])
            foms = np.concatenate([foms, new_foms])
            history['timing_metrics']['new_data_sim_time'].append(time.time() - t_sim_start)
            
            # 更新全域最佳解
            curr_min = np.min(foms)
            if curr_min < best_fom:
                best_fom = curr_min
                best_config = configs[np.argmin(foms)]
                print(f"  *** Breakthrough! New Best FOM: {best_fom:.4f} ***")
                
            print(f"  Added {num_new_samples} samples. Global Best FOM: {best_fom:.4f}")
        else:
            print("  No new unique configs found. Skipping simulation.")
            
        best_fom_history.append(best_fom)

    # 送出 STOP_TAG 讓所有 Worker 節點結束無限迴圈，釋放伺服器運算資源
    for i in range(1, size): comm.send(None, dest=i, tag=STOP_TAG)
    
    print("\n=== Finalizing ===")
    
    # 輸出最佳化軌跡圖
    plot_fom_history(best_fom_history, output_folder)
    plot_optimization_trajectory(foms, PARAMS['INIT_DATASET_SIZE'], output_folder)
    
    # 展開最佳結構的矩陣並儲存
    if len(best_config) == Q_ROWS * Q_COLS:
        Q = np.array(best_config).reshape((Q_ROWS, Q_COLS))
        config_NxN = enforce_reflection_symmetry(Q)
    else:
        config_NxN = np.array(best_config).reshape((FULL_ROWS, FULL_COLS))
        
    final_config_NxN = config_NxN.T 
    np.save(os.path.join(output_folder, f"best_config_{FULL_ROWS}x{FULL_COLS}.npy"), final_config_NxN)
    
    # 繪製最佳二元結構圖
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(final_config_NxN, cmap='gray_r', origin='lower', 
              extent=[-MDM_LC/2, MDM_LC/2, -MDM_LC/2, MDM_LC/2]) 
    
    ax.set_xticks(np.linspace(-MDM_LC/2, MDM_LC/2, FULL_COLS+1), minor=True)
    ax.set_yticks(np.linspace(-MDM_LC/2, MDM_LC/2, FULL_ROWS+1), minor=True)
    ax.grid(which='minor', color='black', linestyle='-', linewidth=1.5)
    ax.tick_params(which='minor', size=0) 
    ax.set_axisbelow(False)
    
    ax.set_title(f"Best Binary Configuration ({FULL_ROWS}x{FULL_COLS})", fontsize=16)
    ax.set_xlabel("X ($\mu$m)", fontsize=14)
    ax.set_ylabel("Y ($\mu$m)", fontsize=14)
    plt.savefig(os.path.join(output_folder, f"best_config_{FULL_ROWS}x{FULL_COLS}.png"), dpi=300)
    plt.close()

    for i in range(1, size): comm.send(None, dest=i, tag=STOP_TAG)

    # 對最佳結構執行完整的 FDTD 場域分析    
    t_final_meep_start = time.time()
    final_res = perform_detailed_final_analysis(best_config, FULL_ROWS, FULL_COLS, output_folder)
    total_meep_time += time.time() - t_final_meep_start
    total_run_time = time.time() - start_total
    total_fm = sum(history['timing_metrics']['fm_train_time'])
    total_sa = sum(history['timing_metrics']['sa_time'])
    total_fdtd = total_meep_time
    other = total_run_time - (total_fm + total_sa + total_fdtd)
    plot_time_statistics_pie_chart(total_fm, total_sa, total_fdtd, other, output_folder)

    # 彙整所有超參數、時間成本與最終結果寫入 JSON，確保實驗具有可重複性
    time_records = {
        "total_time_seconds": total_run_time,
        "total_meep_time_seconds": total_meep_time, 
        "avg_fm_train_time": float(np.mean(history['timing_metrics']['fm_train_time'])) if history['timing_metrics']['fm_train_time'] else 0,
        "avg_new_data_sim_time": float(np.mean(history['timing_metrics']['new_data_sim_time'])) if history['timing_metrics']['new_data_sim_time'] else 0,
        "total_fm_time": total_fm,
        "total_sa_time": total_sa,
        "total_fdtd_time": total_fdtd,
        "other_time": other
    }
    
    experiment_log = {
        "timestamp": timestamp,
        "hyperparameters": {
            "RESOLUTION": RESOLUTION,
            "MDM_LC": MDM_LC,
            "WG_LENGTH": WG_LENGTH,
            "WG_WIDTH": WG_WIDTH,
            "N_SIO2": N_SIO2,
            "N_SI": N_SI,
            "wl_cen": wl_cen,
            "FCEN": FCEN,
            "DF": DF,
            "NFREQ": NFREQ,
            "gaussian_kernel_size": KERNEL_SIZE,     
            "gaussian_kernel_sigma": KERNEL_SIGMA,   
            "tanh_beta": TANH_BETA,                  
            "tanh_eta": TANH_ETA,                    
            "smooth_threshold": SMOOTH_THRESHOLD,    
            "GRID_ROWS": FULL_ROWS,
            "GRID_COLS": FULL_COLS,
            "SAMPLER_TYPE": PARAMS['SAMPLER_TYPE'],
            "INIT_SIM_COUNT": PARAMS['INIT_DATASET_SIZE'],
            "ITERATIONS": PARAMS['ITERATIONS'],
            "SAMPLES_PER_ITER": PARAMS['ADDING_NUM'],
            "FM_EPOCHS": PARAMS['NUM_EPOCHS'],
            "FM_LR": PARAMS['LEARNING_RATE'],
            "FM_K": PARAMS['K_FACTOR'],
            "NUM_READS": PARAMS['NUM_READS'],
            "NUM_SWEEPS": NUM_SWEEPS,
        },
        "time_records": time_records,
        "results": {
            "best_fom": float(best_fom), 
            "best_latent_flat": best_config.tolist(), 
            "fom_evolution_history": best_fom_history,
            "all_evaluated_foms": foms.tolist(),
            "final_analysis": final_res
        }
    }
    
    for mode in final_res:
        for wl_key, vals in final_res[mode].items():
            experiment_log["results"][f"{mode} Through @ {wl_key}"] = f"{vals[0]:.4f} ({vals[1]:.2f} dB)"
    
    with open(os.path.join(output_folder, "final_result.json"), 'w') as f:
        json.dump(experiment_log, f, indent=4)
        
    print(f"Done. Results in {output_folder}")

if __name__ == "__main__":
    if size > 1 and rank != 0: worker_node()
    else: master_node()